# Concordance-field transfer test: do the bright held-out clouds re-center?

**Question (2026-07-08).** The paper makes three claims that must commute:
1. Fig. 7 caption / §5: bright (S/N≥30) offset clouds sit at a coherent ~7–10 mas offset, "most of it a common shift shared across all ten bands" — identified as the concordance field.
2. Fig. 10 / §5: the HGP decomposition is dominated by a ~7 mas common component.
3. §5 close: "the concordance field is a measurement, not a correction: applying the fitted field to held-out positions does not improve them."

If (1) is literally right, subtracting the fitted field from **held-out bright** sources must visibly re-center their clouds; claim (3) as stated was only ever tested **pooled over all sources**, where a ~7 mas coherent shift under ~50 mas per-source scatter is unmeasurable (quadrature change < 1 mas).

**Design.** Refit the production HGP (identical args to `run_q1_astrometry_downstream.sh` step 4) on the dedup Q1 raw anchors **excluding patch 25** (the project's canonical held-out patch, ~13% of anchors, contiguous sky). Then evaluate patch-25 sources against that field: per-band bright cloud centers and median radial offsets, before vs after subtracting (a) the full per-band posterior field, (b) the COMMON component alone. Also the pooled all-source numbers, to quantify the insensitivity of the original test.

Anchors/products: `models/checkpoints/latent_position_q1_vissep/`. Outputs: `io/_nb19_outputs/`.

In [1]:
from pathlib import Path
import subprocess, sys
import numpy as np

REPO = Path('/home/shemmati/Work/Projects/JAISP')
DIR  = REPO / 'models/checkpoints/latent_position_q1_vissep'
DEDUP = DIR / 'anchors_centernet_q1_vissep_dedup.npz'
TRAIN_NPZ = DIR / 'anchors_centernet_q1_vissep_dedup_nopatch25.npz'
FIELD_FITS = DIR / 'hgp_q1_vissep_raw_supertight_nopatch25.fits'
OUT = REPO / 'io/_nb19_outputs'
BANDS = ['u','g','r','i','z','y','nisp_Y','nisp_J','nisp_H']
HOLD_PATCH = '25'

full = dict(np.load(DEDUP, allow_pickle=True))

def patch_of(tiles):
    return np.array([t.split('patch_')[-1] for t in tiles])

# --- build the patch-25-excluded training anchor file (skip if present) ---
if not TRAIN_NPZ.exists():
    train = {}
    for b in BANDS:
        keep = patch_of(full[b+'_tiles']) != HOLD_PATCH
        for suf in ['_ra','_dec','_raw','_head_resid','_snr','_tiles']:
            train[b+suf] = full[b+suf][keep]
        print(f'{b}: kept {keep.sum()}/{len(keep)}')
    np.savez(TRAIN_NPZ, **train)
print('train file:', TRAIN_NPZ.name, TRAIN_NPZ.exists())

train file: anchors_centernet_q1_vissep_dedup_nopatch25.npz True


In [2]:
# --- refit HGP without patch 25, identical production args (driver step 4) ---
if not FIELD_FITS.exists():
    cmd = ['python','-u','models/astrometry2/fit_hierarchical_gp_concordance.py',
        '--anchors', str(TRAIN_NPZ), '--output', str(FIELD_FITS),
        '--offset-kind','raw','--bands', ','.join(BANDS),
        '--length-scales','300,900','--max-centers-per-scale','120',
        '--prior-common-mas','4.0','--prior-group-mas','2.0','--prior-band-mas','1.0',
        '--robust-iters','3','--huber-k','3.0','--dstep-arcsec','5.0',
        '--holdout-mode','spatial','--save-components','--write-coverage','--seed','42']
    r = subprocess.run(cmd, cwd=REPO, env={'PYTHONPATH':'models','PATH':'/usr/bin:/bin:/usr/local/bin','HOME':'/home/shemmati'},
                       capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-2000:], file=sys.stderr)
    assert FIELD_FITS.exists(), 'HGP refit failed'
print('field:', FIELD_FITS.name)

field: hgp_q1_vissep_raw_supertight_nopatch25.fits


In [3]:
# --- sample the held-out field at patch-25 anchor positions ---
from astropy.io import fits
from astropy.wcs import WCS
from scipy.ndimage import map_coordinates
import warnings

hdul = fits.open(FIELD_FITS)
w = WCS(hdul['COMP_COMMON_DRA'].header)   # all maps share the WCS

def sample(ext, ra, dec):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        x, y = w.world_to_pixel_values(ra, dec)
    d = hdul[ext].data
    return map_coordinates(d, [np.atleast_1d(y), np.atleast_1d(x)], order=1, mode='nearest')

def hduname(b):  # band -> extension prefix
    return b.upper() if not b.startswith('nisp') else 'NISP_'+b.split('_')[1]

res = {}
for b in BANDS:
    m25 = patch_of(full[b+'_tiles']) == HOLD_PATCH
    ra, dec = full[b+'_ra'][m25], full[b+'_dec'][m25]
    raw = full[b+'_raw'][m25] * 1000.0          # arcsec -> mas
    snr = full[b+'_snr'][m25]
    fld = np.stack([sample(hduname(b)+'.DRA', ra, dec),
                    sample(hduname(b)+'.DDE', ra, dec)], axis=1) * 1000.0
    com = np.stack([sample('COMP_COMMON_DRA', ra, dec),
                    sample('COMP_COMMON_DDE', ra, dec)], axis=1) * 1000.0
    res[b] = dict(raw=raw, after_common=raw-com, after_full=raw-fld, snr=snr)
print({b: len(v['raw']) for b, v in res.items()})

{'u': 477, 'g': 2652, 'r': 3107, 'i': 2751, 'z': 1835, 'y': 707, 'nisp_Y': 3673, 'nisp_J': 4142, 'nisp_H': 4046}


In [4]:
# --- metrics: cloud centers + median radial offsets, bright and all ---
def center(o):  return np.median(o, axis=0)
def medrad(o):  return np.median(np.hypot(o[:,0], o[:,1]))
def cerr(o):    # bootstrap-free center error: 1.253*MAD/sqrt(n) per axis
    mad = np.median(np.abs(o - np.median(o,axis=0)), axis=0) * 1.4826
    return 1.253 * mad / np.sqrt(len(o))

rows = []
def add(name, sel_bands, cut):
    for kind in ['raw','after_common','after_full']:
        o = np.concatenate([res[b][kind][res[b]['snr'] >= cut] for b in sel_bands]) if cut else             np.concatenate([res[b][kind] for b in sel_bands])
        rows.append((name, kind, len(o), *center(o), np.hypot(*center(o)), *cerr(o), medrad(o)))

for b in BANDS: add(b, [b], 30)
add('RUBIN(pool)',  ['u','g','r','i','z','y'], 30)
add('NISP(pool)',   ['nisp_Y','nisp_J','nisp_H'], 30)
add('ALL bright',   BANDS, 30)
add('ALL sources',  BANDS, 0)

hdr = f"{'set':<12} {'kind':<13} {'n':>6} {'cRA':>7} {'cDec':>7} {'|c|':>6} {'eRA':>5} {'eDec':>5} {'medR':>7}"
print(hdr); print('-'*len(hdr))
for r_ in rows:
    print(f"{r_[0]:<12} {r_[1]:<13} {r_[2]:>6d} {r_[3]:>7.1f} {r_[4]:>7.1f} {r_[5]:>6.1f} {r_[6]:>5.1f} {r_[7]:>5.1f} {r_[8]:>7.1f}")
print('\n(all mas; c=cloud center, e=center std err, medR=median radial offset from origin)')
import json as _json
_json.dump([dict(zip(['set','kind','n','cra','cdec','cmag','era','edec','medrad'], r_)) for r_ in rows],
           open(OUT/'field_transfer_metrics.json','w'), indent=1, default=float)

set          kind               n     cRA    cDec    |c|   eRA  eDec    medR
----------------------------------------------------------------------------
u            raw               12    -9.5    22.9   24.7   3.5   5.1    29.9
u            after_common      12    -8.9    20.3   22.1   4.5   4.5    26.3
u            after_full        12    -6.1    17.3   18.3   4.4   4.3    23.2
g            raw              155     3.2     1.7    3.6   1.6   1.3    15.9
g            after_common     155     4.4    -0.3    4.4   1.8   1.4    18.0
g            after_full       155     9.8    -4.4   10.7   1.7   1.4    19.2
r            raw              247     4.7     1.4    4.9   0.8   0.7    11.5
r            after_common     247     6.6    -1.3    6.7   0.9   0.8    13.8
r            after_full       247     8.0    -1.8    8.1   1.0   0.8    14.6
i            raw              257     6.1     2.3    6.5   0.6   0.6    11.2
i            after_common     257     7.7    -0.6    7.7   0.8   0.6    13.3

In [5]:
# --- figure: bright held-out clouds before/after field subtraction ---
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(11, 11), sharex=True, sharey=True)
for ax, b in zip(axes.ravel(), BANDS):
    sel = res[b]['snr'] >= 30
    for kind, col, lab in [('raw','#1f77b4','raw'), ('after_common','#2ca02c','- common'), ('after_full','#d62728','- full field')]:
        o = res[b][kind][sel]
        c = center(o); r50 = np.median(np.hypot(*(o-c).T))
        ax.scatter(o[:,0], o[:,1], s=4, alpha=0.18, color=col)
        ax.add_patch(plt.Circle(c, r50, fill=False, color=col, lw=1.8, label=f'{lab}: |c|={np.hypot(*c):.1f} mas'))
        ax.plot(*c, marker='x', ms=9, mew=2.5, color=col)
    ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
    ax.set_xlim(-80, 80); ax.set_ylim(-80, 80); ax.set_aspect('equal')
    ax.set_title(f'{b}  (n={sel.sum()})', fontsize=11)
    ax.legend(fontsize=7, loc='lower left', framealpha=0.85)
for ax in axes[-1]: ax.set_xlabel('dRA* [mas]')
for ax in axes[:,0]: ax.set_ylabel('dDec [mas]')
fig.suptitle('Held-out patch 25, bright (S/N$\geq$30): offset clouds before/after subtracting the\n'
             'HGP field fit on the other 5 patches (X = cloud center, circle = 50% radius)', y=0.995)
fig.tight_layout()
fig.savefig(OUT/'field_transfer_brightclouds.png', dpi=150)
print('saved', OUT/'field_transfer_brightclouds.png')
plt.show()

saved /home/shemmati/Work/Projects/JAISP/io/_nb19_outputs/field_transfer_brightclouds.png


In [6]:
# --- verdict ---
import pandas as pd
df = pd.DataFrame([dict(zip(['set','kind','n','cra','cdec','cmag','era','edec','medrad'], r_)) for r_ in rows])
piv = df.pivot_table(index='set', columns='kind', values='cmag')
brightsets = piv.loc[['ALL bright','RUBIN(pool)','NISP(pool)']]
print('|center| (mas):'); print(brightsets[['raw','after_common','after_full']].round(1))
allsrc = df[df['set']=='ALL sources'].set_index('kind')
print('\nALL sources pooled median radial (mas):',
      {k: round(allsrc.loc[k,'medrad'],1) for k in ['raw','after_common','after_full']})
ok_common = (brightsets['after_common'] < 0.55*brightsets['raw']).all()
ok_full   = (brightsets['after_full']  < 0.55*brightsets['raw']).all()
print('\nVERDICT: common re-centers bright held-out clouds:', bool(ok_common),
      '| full field re-centers:', bool(ok_full))

|center| (mas):
kind         raw  after_common  after_full
set                                       
ALL bright   4.7           6.2         7.7
RUBIN(pool)  4.5           6.3         8.1
NISP(pool)   7.0           6.9         4.3

ALL sources pooled median radial (mas): {'raw': 50.6, 'after_common': 50.9, 'after_full': 51.0}

VERDICT: common re-centers bright held-out clouds: False | full field re-centers: False


## Follow-up: is the bright offset spatially variable, and does the field describe the bright population even in-sample?

The transfer test failed, but with a twist: the held-out bright clouds sit at ~4.5 mas (Rubin pool), not the caption's 7-10 mas. Two discriminating checks:
1. **Per-patch bright cloud centers (raw)** — does the "coherent bright offset" vary across the field?
2. **In-sample test** — subtract the *production* field (fit on ALL patches, `hgp_q1_vissep_raw_supertight.fits`) from bright sources everywhere. If even the in-sample field fails to re-center bright clouds, the field is a property of the faint-dominated anchor population, not of the bright clouds — the caption's causal identification is overclaimed regardless of transferability.

In [7]:
# --- per-patch bright cloud centers (raw), pooled over bands ---
PROD_FITS = DIR / 'hgp_q1_vissep_raw_supertight.fits'
hdul_prod = fits.open(PROD_FITS)
w_prod = WCS(hdul_prod['COMP_COMMON_DRA'].header)

def sample_prod(ext, ra, dec):
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        x, y = w_prod.world_to_pixel_values(ra, dec)
    return map_coordinates(hdul_prod[ext].data, [np.atleast_1d(y), np.atleast_1d(x)], order=1, mode='nearest')

patches = ['4','5','14','15','24','25']
print(f"{'patch':<7} {'n_bright':>8} {'cRA':>7} {'cDec':>7} {'|c|':>6} | after prod field: {'cRA':>7} {'cDec':>7} {'|c|':>6}")
summary_pp = {}
for pch in patches:
    raws, afters = [], []
    for b in BANDS:
        pm = patch_of(full[b+'_tiles']) == pch
        bm = pm & (full[b+'_snr'] >= 30)
        if bm.sum() == 0: continue
        ra, dec = full[b+'_ra'][bm], full[b+'_dec'][bm]
        raw = full[b+'_raw'][bm] * 1000.0
        fld = np.stack([sample_prod(hduname(b)+'.DRA', ra, dec),
                        sample_prod(hduname(b)+'.DDE', ra, dec)], axis=1) * 1000.0
        raws.append(raw); afters.append(raw - fld)
    R, A = np.concatenate(raws), np.concatenate(afters)
    cr, ca = center(R), center(A)
    summary_pp[pch] = dict(n=len(R), craw=list(map(float,cr)), cafter=list(map(float,ca)))
    print(f"{pch:<7} {len(R):>8d} {cr[0]:>7.1f} {cr[1]:>7.1f} {np.hypot(*cr):>6.1f} |"
          f" {'':<17} {ca[0]:>7.1f} {ca[1]:>7.1f} {np.hypot(*ca):>6.1f}")

# field-wide pooled bright, in-sample
Rall = np.concatenate([full[b+'_raw'][full[b+'_snr']>=30]*1000.0 for b in BANDS])
print(f"\nfield-wide pooled bright: n={len(Rall)}, center=({center(Rall)[0]:.1f},{center(Rall)[1]:.1f}), |c|={np.hypot(*center(Rall)):.1f} mas, medR={medrad(Rall):.1f} mas")

patch   n_bright     cRA    cDec    |c| | after prod field:     cRA    cDec    |c|
4           1393    -0.7    12.6   12.6 |                       0.6     0.8    1.0
5           1621     2.0     8.8    9.0 |                       1.5     1.8    2.4


14          1426     0.3    10.4   10.4 |                       0.4     1.6    1.7


15          1372     4.3     4.5    6.3 |                       0.3     0.9    1.0
24          1262     2.3     5.1    5.6 |                      -0.3     0.4    0.5
25          1095     4.3     1.8    4.7 |                       2.0     0.3    2.0

field-wide pooled bright: n=8169, center=(1.9,7.6), |c|=7.8 mas, medR=14.1 mas


In [8]:
# --- in-sample: production field subtracted from ALL bright sources, per band and pooled ---
rows2 = []
for b in BANDS:
    bm = full[b+'_snr'] >= 30
    ra, dec = full[b+'_ra'][bm], full[b+'_dec'][bm]
    raw = full[b+'_raw'][bm] * 1000.0
    fld = np.stack([sample_prod(hduname(b)+'.DRA', ra, dec),
                    sample_prod(hduname(b)+'.DDE', ra, dec)], axis=1) * 1000.0
    com = np.stack([sample_prod('COMP_COMMON_DRA', ra, dec),
                    sample_prod('COMP_COMMON_DDE', ra, dec)], axis=1) * 1000.0
    rows2.append((b, len(raw), np.hypot(*center(raw)), np.hypot(*center(raw-com)), np.hypot(*center(raw-fld)),
                  medrad(raw), medrad(raw-fld)))
Rall_r, Rall_c, Rall_f = [], [], []
for b in BANDS:
    bm = full[b+'_snr'] >= 30
    ra, dec = full[b+'_ra'][bm], full[b+'_dec'][bm]
    raw = full[b+'_raw'][bm]*1000.0
    fld = np.stack([sample_prod(hduname(b)+'.DRA', ra, dec), sample_prod(hduname(b)+'.DDE', ra, dec)],axis=1)*1000.0
    com = np.stack([sample_prod('COMP_COMMON_DRA', ra, dec), sample_prod('COMP_COMMON_DDE', ra, dec)],axis=1)*1000.0
    Rall_r.append(raw); Rall_c.append(raw-com); Rall_f.append(raw-fld)
Rr, Rc, Rf = map(np.concatenate, (Rall_r, Rall_c, Rall_f))
print(f"{'band':<8} {'n':>6} {'|c| raw':>8} {'-common':>8} {'-full':>7} {'medR raw':>9} {'medR -full':>10}")
for r_ in rows2:
    print(f"{r_[0]:<8} {r_[1]:>6d} {r_[2]:>8.1f} {r_[3]:>8.1f} {r_[4]:>7.1f} {r_[5]:>9.1f} {r_[6]:>10.1f}")
print(f"{'ALL':<8} {len(Rr):>6d} {np.hypot(*center(Rr)):>8.1f} {np.hypot(*center(Rc)):>8.1f} {np.hypot(*center(Rf)):>7.1f} {medrad(Rr):>9.1f} {medrad(Rf):>10.1f}")
print('\nIN-SAMPLE verdict: production field re-centers pooled bright clouds:',
      bool(np.hypot(*center(Rf)) < 0.55*np.hypot(*center(Rr))))

band          n  |c| raw  -common   -full  medR raw medR -full
u           133     22.6     17.9    18.2      28.2       23.7
g          1171      7.9      2.2     5.0      16.5       14.9
r          1820      7.9      1.9     1.5      14.2        9.6
i          1927      9.0      3.0     1.8      13.4        8.5
z          1437      5.3      1.6     0.4       9.8        7.2
y           483      9.0      4.1     3.5      11.5        7.3
nisp_Y      316     12.6      6.1     4.0      25.9       18.9
nisp_J      441      9.2      4.0     2.3      19.1       16.2
nisp_H      441      8.2      1.9     2.3      21.0       18.1
ALL        8169      7.8      1.6     1.2      14.1       10.5

IN-SAMPLE verdict: production field re-centers pooled bright clouds: True


## Interpolation vs extrapolation: bright sources in the solver's own 300" holdout blocks

The whole-patch test above is *extrapolation* (hole size ≳ correlation lengths of 300"/900"). The production fit's own spatial holdout uses scattered 300" blocks — *interpolation*. If bright sources inside those small holes DO re-center, the correct statement for §5 is that the field transfers at short range (interpolation) but not to unfitted sky (extrapolation), and "applying the fitted field to held-out positions does not improve them" needs its geometry qualified.

In [9]:
# --- replicate production fit with its own spatial 300"-block holdout; evaluate bright holdout sources ---
import sys
sys.path.insert(0, str(REPO/'models'))
from astrometry2 import fit_hierarchical_gp_concordance as H

argv = ['--anchors', str(DEDUP), '--output', '/tmp/unused.fits',
        '--offset-kind','raw','--bands', ','.join(BANDS),
        '--length-scales','300,900','--max-centers-per-scale','120',
        '--prior-common-mas','4.0','--prior-group-mas','2.0','--prior-band-mas','1.0',
        '--robust-iters','3','--huber-k','3.0','--dstep-arcsec','5.0',
        '--holdout-mode','spatial','--seed','42']
args = H.build_argparser().parse_args(argv)
bands_n = [H.normalize_band_name(b) for b in BANDS]
anch = H.load_anchor_cache(cache_path=DEDUP, bands=bands_n, offset_kind='raw', pool=args.pool,
    snr_min=float(args.snr_min), snr_classical=float(args.snr_classical), clip_mas=float(args.clip_mas),
    clip_sigma=float(args.clip_sigma), snr_weight_floor=float(args.snr_weight_floor),
    snr_weight_cap=float(args.snr_weight_cap), snr_weight_power=float(args.snr_weight_power),
    min_sigma_mas=float(args.min_sigma_mas), max_sigma_mas=float(args.max_sigma_mas),
    dedup_radius_arcsec=float(args.dedup_radius_arcsec), max_anchors=None, seed=42)
basis = H.make_basis(anch.pos_shifted, length_scales=[300.0,900.0], spacing_factor=float(args.spacing_factor),
    max_centers_per_scale=120, support_factor=float(args.support_factor))
hier = H.make_hierarchy(n_base=int(basis.centers.shape[0]), band_names=anch.band_names,
    use_common=True, use_group=True, use_band=True,
    prior_common_mas=4.0, prior_group_mas=2.0, prior_band_mas=1.0)
hmask = H.make_holdout_mask(anch.pos_shifted, frac=0.10, mode='spatial', block_arcsec=300.0, seed=42)
print(f'holdout {hmask.sum():,}/{len(hmask):,}')
model = H.fit_hgp(anch, basis, hier, support_factor=float(args.support_factor), train_mask=~hmask,
                  robust_iters=3, huber_k=3.0, batch_size=int(args.batch_size), jitter=float(args.jitter))

  u      :    3,628 anchors, med |offset|=122.2 mas, MADxy= 99.0 mas, SNR_ref= 10.7
  g      :   17,626 anchors, med |offset|= 56.0 mas, MADxy= 45.4 mas, SNR_ref=  8.8
  r      :   20,624 anchors, med |offset|= 48.0 mas, MADxy= 38.9 mas, SNR_ref=  9.3
  i      :   18,270 anchors, med |offset|= 43.9 mas, MADxy= 34.7 mas, SNR_ref=  9.5
  z      :   12,622 anchors, med |offset|= 44.4 mas, MADxy= 34.9 mas, SNR_ref=  9.8
  y      :    5,059 anchors, med |offset|= 66.2 mas, MADxy= 49.3 mas, SNR_ref= 11.0
  nisp_Y :   23,386 anchors, med |offset|= 52.2 mas, MADxy= 43.4 mas, SNR_ref=  8.8
  nisp_J :   26,762 anchors, med |offset|= 49.0 mas, MADxy= 40.1 mas, SNR_ref=  9.3
  nisp_H :   26,047 anchors, med |offset|= 49.0 mas, MADxy= 40.0 mas, SNR_ref=  9.5

Loaded 154,024 anchors from /home/shemmati/Work/Projects/JAISP/models/checkpoints/latent_position_q1_vissep/anchors_centernet_q1_vissep_dedup.npz
  offset kind       : raw
  pool              : all
  median |offset|   : 50.48 mas
  field exten

    robust downweight: 14,922/135,609 (11.0%)
  HGP solve iteration 2/3


    robust downweight: 14,913/135,609 (11.0%)
  HGP solve iteration 3/3


  train residual: med=49.53 mas, MADxy=40.60 mas


In [10]:
# bright (S/N>=30) sources inside the 300" holdout blocks: before/after field prediction
snr_all = anch.snr  # anchor-table SNR
bright_h = hmask & (snr_all >= 30)
idx = np.where(bright_h)[0]
mean, _ = H.predict(model, anch.pos_shifted[idx], anch.group_idx[idx], anch.band_name_idx[idx],
                 float(args.support_factor), int(args.batch_size))
raw_h = anch.offsets[idx] * 1000.0
aft_h = (anch.offsets[idx] - mean) * 1000.0
print(f'n bright in 300" holdout blocks: {len(idx)}')
print(f'raw   center=({center(raw_h)[0]:.1f},{center(raw_h)[1]:.1f}) |c|={np.hypot(*center(raw_h)):.1f} mas, medR={medrad(raw_h):.1f}')
print(f'after center=({center(aft_h)[0]:.1f},{center(aft_h)[1]:.1f}) |c|={np.hypot(*center(aft_h)):.1f} mas, medR={medrad(aft_h):.1f}')
print('\nINTERPOLATION verdict: field re-centers bright sources in small held-out holes:',
      bool(np.hypot(*center(aft_h)) < 0.55*np.hypot(*center(raw_h))))

n bright in 300" holdout blocks: 962
raw   center=(1.2,9.3) |c|=9.4 mas, medR=14.0
after center=(0.6,0.5) |c|=0.8 mas, medR=10.3

INTERPOLATION verdict: field re-centers bright sources in small held-out holes: True
